In [ ]:
# ===================== IMPORTS =====================
import os, cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ConvNeXtLarge
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

# ===================== CONFIG =====================
DATASET_DIR = "/kaggle/input/mangoleafds224x224/MangoLeafDS224x224"
IMG_SIZE = 224
BATCH_SIZE = 16        # ConvNeXt-Large is heavy
EPOCHS = 15
N_SPLITS = 5
NUM_CLASSES = 6
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ===================== LOAD IMAGE PATHS =====================
image_paths, labels = [], []
class_names = sorted(os.listdir(DATASET_DIR))

for idx, cls in enumerate(class_names):
    for f in os.listdir(os.path.join(DATASET_DIR, cls)):
        image_paths.append(os.path.join(DATASET_DIR, cls, f))
        labels.append(idx)

image_paths = np.array(image_paths)
labels = np.array(labels)

# ===================== DATA GENERATOR =====================
def data_generator(paths, labels):
    while True:
        idxs = np.arange(len(paths))
        np.random.shuffle(idxs)

        for i in range(0, len(paths), BATCH_SIZE):
            batch_idx = idxs[i:i+BATCH_SIZE]
            imgs, labs = [], []

            for j in batch_idx:
                img = cv2.imread(paths[j])
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                imgs.append(img)
                labs.append(labels[j])

            yield np.array(imgs)/255.0, tf.keras.utils.to_categorical(labs, NUM_CLASSES)

# ===================== MODEL =====================
def build_model():
    base = ConvNeXtLarge(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False

    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = models.Model(base.input, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ===================== 5-FOLD CV =====================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(image_paths, labels), 1):
    print(f"\n========== FOLD {fold} ==========")

    model = build_model()

    train_gen = data_generator(image_paths[train_idx], labels[train_idx])
    val_gen   = data_generator(image_paths[val_idx], labels[val_idx])

    model.fit(
        train_gen,
        steps_per_epoch=len(train_idx)//BATCH_SIZE,
        validation_data=val_gen,
        validation_steps=len(val_idx)//BATCH_SIZE,
        epochs=EPOCHS,
        verbose=1
    )

    _, acc = model.evaluate(val_gen, steps=len(val_idx)//BATCH_SIZE, verbose=0)
    print(f"Fold {fold} Accuracy: {acc:.4f}")
    fold_accuracies.append(acc)

# ===================== RESULTS =====================
print("\nMean Accuracy:", np.mean(fold_accuracies))
print("Std Accuracy :", np.std(fold_accuracies))

plt.plot(range(1, N_SPLITS+1), fold_accuracies, marker='o')
plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.title("ConvNeXt-Large (5-Fold CV)")
plt.grid(True)
plt.show()
